In [1]:
# =========================================
# 1. Install (run once)
# =========================================
# !pip install librosa pandas numpy torch torchvision scikit-learn tqdm


# =========================================
# 2. Imports
# =========================================
import os
import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import torchvision.models as models


# =========================================
# 3. Paths
# =========================================
DATA_PATH = r"C:\ML_MCA\wildlife-audio-project\data\train_audio"
METADATA_PATH = r"C:\ML_MCA\wildlife-audio-project\data\taxonomy.csv"


# =========================================
# 4. Load Metadata
# =========================================
metadata = pd.read_csv(METADATA_PATH)

file_paths = []
labels = []

for _, row in metadata.iterrows():

    species = row["primary_label"]
    filename = row["filename"]

    file_path = os.path.join(DATA_PATH, species, filename)

    if os.path.exists(file_path):
        file_paths.append(file_path)
        labels.append(species)

print("Total files:", len(file_paths))


# =========================================
# 5. Encode Labels
# =========================================
le = LabelEncoder()
labels = le.fit_transform(labels)

num_classes = len(set(labels))


# =========================================
# 6. Train-Test Split
# =========================================
train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, labels, test_size=0.2, random_state=42
)


# =========================================
# 7. Audio Processing Functions
# =========================================
def split_audio(file_path, duration=5, sr=32000):

    audio, sr = librosa.load(file_path, sr=sr)

    samples_per_segment = sr * duration

    segments = []

    for start in range(0, len(audio), samples_per_segment):

        segment = audio[start:start + samples_per_segment]

        if len(segment) == samples_per_segment:
            segments.append(segment)

    return segments


def audio_to_mel(segment, sr=32000):

    mel = librosa.feature.melspectrogram(y=segment, sr=sr, n_mels=128)

    mel = librosa.power_to_db(mel)

    return mel


# =========================================
# 8. Dataset Class (IMPORTANT)
# =========================================
class BirdDataset(Dataset):

    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = labels

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):

        segments = split_audio(self.file_paths[idx])

        if len(segments) == 0:
            segment = np.zeros(32000 * 5)
        else:
            segment = segments[0]

        mel = audio_to_mel(segment)

        mel = torch.tensor(mel).float()

        # Convert to 3 channels (ResNet input)
        mel = mel.unsqueeze(0).repeat(3,1,1)

        label = torch.tensor(self.labels[idx])

        return mel, label


# =========================================
# 9. DataLoaders
# =========================================
train_dataset = BirdDataset(train_paths, train_labels)
val_dataset = BirdDataset(val_paths, val_labels)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


# =========================================
# 10. Load ResNet50
# =========================================
model = models.resnet50(pretrained=True)

model.fc = nn.Linear(model.fc.in_features, num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)


# =========================================
# 11. Training Setup
# =========================================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)


# =========================================
# 12. Training Loop
# =========================================
EPOCHS = 3

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for x, y in tqdm(train_loader):

        x = x.to(device)
        y = y.to(device)

        outputs = model(x)

        loss = criterion(outputs, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss}")


# =========================================
# 13. Evaluation (Accuracy)
# =========================================
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for x, y in val_loader:

        x = x.to(device)
        y = y.to(device)

        outputs = model(x)

        _, preds = torch.max(outputs, 1)

        total += y.size(0)
        correct += (preds == y).sum().item()

accuracy = 100 * correct / total

print("\n==============================")
print("Validation Accuracy:", round(accuracy, 2), "%")
print("==============================")

KeyError: 'filename'

In [ ]:
import os

DATA_PATH = r"C:\ML_MCA\wildlife-audio-project\data"

for root, dirs, files in os.walk(DATA_PATH):
    print("Folder:", root)
    print("Subfolders:", dirs[:5])
    print("Files:", files[:5])
    print("-"*50)
    break

Folder: C:\ML_MCA\wildlife-audio-project\data
Subfolders: ['test_soundscapes', 'train_audio', 'train_soundscapes']
Files: ['sample_submission.csv', 'taxonomy.csv', 'train.csv', 'train_soundscapes_labels.csv']
--------------------------------------------------


In [ ]:
import os
import pandas as pd

DATA_PATH = r"C:\ML_MCA\wildlife-audio-project\data\train_audio"
CSV_PATH = r"C:\ML_MCA\wildlife-audio-project\data\train.csv"

df = pd.read_csv(CSV_PATH)

print(df.head())
print("Total samples:", len(df))

  primary_label secondary_labels type  latitude  longitude scientific_name  \
0       1161364               []   []  -22.7562   -46.8666    Guyalna cuta   
1       1161364               []   []  -22.7558   -46.8700    Guyalna cuta   
2       1161364               []   []  -22.7547   -46.8728    Guyalna cuta   
3       1161364               []   []  -22.7547   -46.8728    Guyalna cuta   
4       1161364               []   []  -22.7426   -46.8985    Guyalna cuta   

    common_name class_name  inat_taxon_id         author   license  rating  \
0  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
1  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
2  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
3  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
4  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   

                                                 url          

In [ ]:
print(df.columns)

Index(['primary_label', 'secondary_labels', 'type', 'latitude', 'longitude',
       'scientific_name', 'common_name', 'class_name', 'inat_taxon_id',
       'author', 'license', 'rating', 'url', 'filename', 'collection'],
      dtype='object')


In [ ]:
file_paths = []
labels = []

for _, row in df.iterrows():
    
    filename = row["filename"]
    label = row["primary_label"]

    path = os.path.join(DATA_PATH, label, filename)

    if os.path.exists(path):
        file_paths.append(path)
        labels.append(label)

print("Loaded files:", len(file_paths))

Loaded files: 0


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
labels = le.fit_transform(labels)

num_classes = len(set(labels))

print("Number of classes:", num_classes)

Number of classes: 0


In [ ]:
import os

DATA_PATH = r"C:\ML_MCA\wildlife-audio-project\data\train_audio"

print(os.listdir(DATA_PATH)[:10])

['1161364', '116570', '1176823', '1595929', '209233', '22930', '22956', '22961', '22967', '22973']


In [ ]:
file_paths = []
labels = []

for _, row in df.iterrows():
    
    filename = row["filename"]
    label = row["primary_label"]

    # FIX: no label folder
    path = os.path.join(DATA_PATH, filename)

    if os.path.exists(path):
        file_paths.append(path)
        labels.append(label)

print("Loaded files:", len(file_paths))

Loaded files: 35549


In [ ]:
# =========================================
# 1. Imports
# =========================================
import os
import librosa
import librosa.display
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
import torchvision.models as models


# =========================================
# 2. Paths
# =========================================
DATA_PATH = r"C:\ML_MCA\wildlife-audio-project\data\train_audio"
CSV_PATH = r"C:\ML_MCA\wildlife-audio-project\data\train.csv"


# =========================================
# 3. Load Dataset
# =========================================
df = pd.read_csv(CSV_PATH)

file_paths = []
labels = []

for _, row in df.iterrows():
    filename = row["filename"]
    label = row["primary_label"]

    path = os.path.join(DATA_PATH, filename)

    if os.path.exists(path):
        file_paths.append(path)
        labels.append(label)

print("Loaded files:", len(file_paths))


# =========================================
# 🎯 VISUALIZATION 1: WAVEFORM
# =========================================
audio, sr = librosa.load(file_paths[0])

plt.figure(figsize=(12,4))
librosa.display.waveshow(audio, sr=sr)
plt.title("Audio Waveform")
plt.show()


# =========================================
# 4. Encode Labels
# =========================================
le = LabelEncoder()
labels = le.fit_transform(labels)

num_classes = len(set(labels))


# =========================================
# 🎯 VISUALIZATION 2: CLASS DISTRIBUTION
# =========================================
label_names = le.inverse_transform(labels)

plt.figure(figsize=(10,4))
pd.Series(label_names).value_counts().head(20).plot(kind='bar')
plt.title("Top 20 Classes")
plt.show()


# =========================================
# 5. Train/Test Split
# =========================================
train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, labels, test_size=0.2, random_state=42
)

# =========================================
# 🎯 VISUALIZATION 3: TRAIN vs VALIDATION
# =========================================
plt.bar(["Train", "Validation"], [len(train_paths), len(val_paths)])
plt.title("Train vs Validation Split")
plt.show()


# =========================================
# 6. Audio Processing Functions
# =========================================
def split_audio(file_path, duration=5, sr=32000):
    audio, sr = librosa.load(file_path, sr=sr)
    samples_per_segment = sr * duration

    segments = []
    for start in range(0, len(audio), samples_per_segment):
        segment = audio[start:start + samples_per_segment]

        if len(segment) == samples_per_segment:
            segments.append(segment)

    return segments


def audio_to_mel(segment, sr=32000):
    mel = librosa.feature.melspectrogram(y=segment, sr=sr, n_mels=128)
    mel = librosa.power_to_db(mel)
    return mel


# =========================================
# 🎯 VISUALIZATION 4: MEL SPECTROGRAM
# =========================================
segment = split_audio(file_paths[0])[0]
mel = audio_to_mel(segment)

plt.figure(figsize=(10,4))
librosa.display.specshow(mel, sr=32000)
plt.colorbar()
plt.title("Mel Spectrogram")
plt.show()


# =========================================
# 7. Dataset Class
# =========================================
class BirdDataset(Dataset):

    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = labels

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):

        segments = split_audio(self.file_paths[idx])

        if len(segments) == 0:
            segment = np.zeros(32000 * 5)
        else:
            segment = segments[0]

        mel = audio_to_mel(segment)

        mel = torch.tensor(mel).float()
        mel = mel.unsqueeze(0).repeat(3,1,1)

        label = torch.tensor(self.labels[idx])

        return mel, label


# =========================================
# 8. DataLoaders
# =========================================
train_dataset = BirdDataset(train_paths, train_labels)
val_dataset = BirdDataset(val_paths, val_labels)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


# =========================================
# 9. Model (ResNet50)
# =========================================
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


# =========================================
# 10. Training Setup
# =========================================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)


# =========================================
# 11. Training Loop + LOSS TRACK
# =========================================
EPOCHS = 3
losses = []

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for x, y in tqdm(train_loader):

        x = x.to(device)
        y = y.to(device)

        outputs = model(x)
        loss = criterion(outputs, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    losses.append(total_loss)

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")


# =========================================
# 🎯 VISUALIZATION 5: LOSS CURVE
# =========================================
plt.plot(losses)
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()


# =========================================
# 12. Evaluation
# =========================================
correct = 0
total = 0

y_true = []
y_pred = []

model.eval()

with torch.no_grad():

    for x, y in val_loader:

        x = x.to(device)
        y = y.to(device)

        outputs = model(x)
        _, preds = torch.max(outputs, 1)

        total += y.size(0)
        correct += (preds == y).sum().item()

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

accuracy = 100 * correct / total

print("\nValidation Accuracy:", round(accuracy, 2), "%")


# =========================================
# 🎯 VISUALIZATION 6: CONFUSION MATRIX
# =========================================
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


# =========================================
# 🎯 VISUALIZATION 7: SAMPLE PREDICTIONS
# =========================================
x, y = next(iter(val_loader))

x = x.to(device)
outputs = model(x)
_, preds = torch.max(outputs, 1)

true_labels = le.inverse_transform(y.numpy())
pred_labels = le.inverse_transform(preds.cpu().numpy())

for i in range(5):
    plt.imshow(x[i][0].cpu(), aspect='auto')
    plt.title(f"True: {true_labels[i]} | Pred: {pred_labels[i]}")
    plt.axis('off')
    plt.show()

ModuleNotFoundError: No module named 'seaborn'

In [1]:
# =========================================
# 1. IMPORTS
# =========================================
import os
import librosa
import librosa.display
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm


# =========================================
# 2. PATHS
# =========================================
DATA_PATH = r"C:\ML_MCA\wildlife-audio-project\data\train_audio"
CSV_PATH = r"C:\ML_MCA\wildlife-audio-project\data\train.csv"


# =========================================
# 3. LOAD DATA
# =========================================
df = pd.read_csv(CSV_PATH)

file_paths = []
labels = []

for _, row in df.iterrows():
    filename = row["filename"]
    label = row["primary_label"]

    path = os.path.join(DATA_PATH, filename)

    if os.path.exists(path):
        file_paths.append(path)
        labels.append(label)

print("Loaded files:", len(file_paths))


# =========================================
# 🎯 VISUALIZATION 1: WAVEFORM
# Shows raw audio amplitude over time
# =========================================
audio, sr = librosa.load(file_paths[0])

plt.figure(figsize=(12,4))
librosa.display.waveshow(audio, sr=sr)
plt.title("Waveform (Audio Signal)")
plt.show()


# =========================================
# 4. LABEL ENCODING
# =========================================
le = LabelEncoder()
labels = le.fit_transform(labels)
num_classes = len(set(labels))


# =========================================
# 🎯 VISUALIZATION 2: CLASS DISTRIBUTION
# Shows dataset imbalance
# =========================================
label_names = le.inverse_transform(labels)

pd.Series(label_names).value_counts().head(20).plot(kind='bar')
plt.title("Top 20 Class Distribution")
plt.show()


# =========================================
# 5. SPLIT DATA
# =========================================
train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, labels, test_size=0.2, random_state=42
)


# =========================================
# 🎯 VISUALIZATION 3: DATA SPLIT
# Shows training vs validation size
# =========================================
plt.bar(["Train", "Validation"], [len(train_paths), len(val_paths)])
plt.title("Train vs Validation Split")
plt.show()


# =========================================
# 6. AUDIO PROCESSING
# =========================================
def split_audio(file_path, duration=5, sr=32000):
    audio, sr = librosa.load(file_path, sr=sr)
    samples = sr * duration

    segments = []
    for start in range(0, len(audio), samples):
        seg = audio[start:start+samples]
        if len(seg) == samples:
            segments.append(seg)
    return segments


def audio_to_mel(segment, sr=32000):
    mel = librosa.feature.melspectrogram(y=segment, sr=sr, n_mels=128)
    mel = librosa.power_to_db(mel)

    # Normalize
    mel = (mel - np.mean(mel)) / np.std(mel)

    return mel


# =========================================
# 🎯 VISUALIZATION 4: MEL SPECTROGRAM
# Shows frequency vs time (what model sees)
# =========================================
seg = split_audio(file_paths[0])[0]
mel = audio_to_mel(seg)

plt.figure(figsize=(10,4))
librosa.display.specshow(mel, sr=32000)
plt.colorbar()
plt.title("Mel Spectrogram")
plt.show()


# =========================================
# 7. DATASET
# =========================================
class BirdDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = paths
        self.labels = labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        segments = split_audio(self.paths[idx])

        if len(segments) == 0:
            seg = np.zeros(32000*5)
        else:
            seg = segments[np.random.randint(len(segments))]

        mel = audio_to_mel(seg)
        mel = torch.tensor(mel).unsqueeze(0).float()

        label = torch.tensor(self.labels[idx])

        return mel, label


# =========================================
# 8. DATALOADER
# =========================================
train_loader = DataLoader(BirdDataset(train_paths, train_labels), batch_size=8, shuffle=True)
val_loader = DataLoader(BirdDataset(val_paths, val_labels), batch_size=8)


# =========================================
# 9. CNN14-STYLE MODEL (SIMPLIFIED)
# =========================================
class CNN14(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,256,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1,1))
        )

        self.fc = nn.Linear(256, num_classes)

    def forward(self,x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


model = CNN14(num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


# =========================================
# 10. TRAINING SETUP
# =========================================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)


# =========================================
# 11. TRAINING
# =========================================
EPOCHS = 5
losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x, y in tqdm(train_loader):
        x, y = x.to(device), y.to(device)

        out = model(x)
        loss = criterion(out, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    losses.append(total_loss)
    print(f"Epoch {epoch+1} Loss: {total_loss:.3f}")


# =========================================
# 🎯 VISUALIZATION 5: LOSS CURVE
# Shows learning progress
# =========================================
plt.plot(losses)
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()


# =========================================
# 12. EVALUATION
# =========================================
y_true, y_pred = [], []

model.eval()

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)

        out = model(x)
        preds = torch.argmax(out, 1)

        y_true.extend(y.numpy())
        y_pred.extend(preds.cpu().numpy())


# =========================================
# 13. ACCURACY + REPORT
# =========================================
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_true, y_pred)
print("\nAccuracy:", round(acc*100,2), "%")

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))


# =========================================
# 🎯 VISUALIZATION 6: CONFUSION MATRIX
# Shows model mistakes
# =========================================
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

KeyboardInterrupt: 